# Enterprise Knowledge Base Assistant — Runnable Baseline

**CSE 598 Capstone, Phase 1**  ·  Pardha Praneeth Pudi

A grounded question-answering baseline over a small internal-documentation corpus.
You ask a work question, it retrieves passages from the company knowledge base and
answers **only** from those passages, citing the exact section it used.

This is the deliberately naive baseline for the capstone: single-pass RAG with
**no agents, no reranking and no access control**, so that the multi-agent system
built in Phase 2 has something honest to be measured against.

---

### How to run this notebook

**Runtime → Run all.** That is the whole procedure.

* No API key is required.
* No model is downloaded.
* No `git clone`, no dataset upload.
* Total runtime is well under a minute on a free Colab CPU runtime.

The notebook writes the entire project to the Colab filesystem (11 corpus
documents, a 32-question gold set, and six pipeline modules), then runs the same
commands the repository README documents:

```
python3 run_baseline.py --input examples/test1.txt
python3 run_eval.py
```

### What you should see

Four test cases — one that works, and three that fail in the specific ways the
multi-agent system is designed to fix — followed by the full 32-question
evaluation.

## 0. Environment

Only three packages, all of which Colab already has. The install line is here so
the notebook also works on a bare Jupyter kernel.

In [1]:
!pip -q install "numpy>=1.24" "PyYAML>=6.0" "scikit-learn>=1.3" 2>/dev/null

import sys, numpy, sklearn, yaml
print("python      :", sys.version.split()[0])
print("numpy       :", numpy.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyYAML      :", yaml.__version__)

python      : 3.11.15
numpy       : 2.4.4
scikit-learn: 1.8.0
PyYAML      : 6.0.3


## 1. The knowledge base

Eleven markdown policy documents for a fictional company, *Northwind Robotics*.
The corpus is synthetic so it can be published with no licensing or
confidentiality problem, but it is written to have the structure that causes real
failures:

* **Cross-referencing sections.** The incident runbook says who to page; the
  security policy says who must approve emergency database access. Neither
  document answers the question on its own.
* **One genuinely restricted document.** `HR-002` (compensation bands) is
  classified `confidential` and contractors are explicitly excluded from it.

Every document carries YAML front matter with `doc_id`, `department` and
`sensitivity`, which is what the access-control evaluation checks against.

In [2]:
import pathlib, textwrap

for d in ("src", "data/corpus", "data/gold", "examples", "outputs"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path("src/__init__.py").write_text("# Baseline RAG pipeline package\n")

CORPUS = {
    'eng-deployment-and-release.md': r'''---
doc_id: ENG-002
title: Deployment and Release Management
department: Engineering
sensitivity: internal
owner: platform@northwind-robotics.example
last_updated: 2026-04-25
---

# Deployment and Release Management

## 1. Release Cadence

Services deploy continuously from `main` behind feature flags. Robot firmware follows
a **fortnightly release train** that cuts on alternate Tuesdays at 10:00 Phoenix time.

## 2. Deployment Freeze Windows

Deployments to production are frozen:

- From **December 18 through January 2** inclusive
- During the **last 3 business days of each fiscal quarter**

Freeze exceptions require **VP of Engineering approval** and are limited to Sev1 and
Sev2 fixes. Feature work is never granted a freeze exception.

## 3. Rollback

Any deployment can be rolled back with `nwctl release rollback --service <name>`.
The rollback target is the last known-good revision. **Rollback must complete within
30 minutes** of a Sev1 being declared, or the incident commander switches to a
forward-fix strategy and says so explicitly on the bridge.

## 4. Firmware Specifics

Robot firmware rollbacks are **staged over 24 hours** and cannot be completed in 30
minutes. Fleet-wide firmware issues therefore follow the forward-fix path by default.

## 5. Change Records

Every production deployment writes a change record automatically from CI. Manual
deployments outside CI are prohibited except under the break-glass procedure in
SEC-001.
''',
    'eng-engineering-onboarding.md': r'''---
doc_id: ENG-001
title: Engineering Onboarding Guide
department: Engineering
sensitivity: internal
owner: platform@northwind-robotics.example
last_updated: 2026-03-30
---

# Engineering Onboarding Guide

## 1. Local Environment

All services build with `make bootstrap`, which installs the pinned toolchain
(Python 3.11, Go 1.23, Node 20) through `mise`. The monorepo is `northwind/core`.
A full clean build takes roughly **12 minutes** on Standard tier hardware.

## 2. Repository Layout

- `platform/` - shared infrastructure, service mesh, auth
- `fleet/` - robot fleet management APIs
- `perception/` - vision and sensor fusion
- `console/` - customer-facing web console

## 3. Code Review

Every change lands through a pull request. **Changes under `platform/` require 2
approvals; all other paths require 1 approval.** CI must be green before merge; there
is no override path for a red build. Pull requests older than **10 days** without
activity are closed automatically.

## 4. Testing

Unit tests run on every push. Integration tests run on merge to `main`. The fleet
simulation suite runs nightly and takes about 40 minutes.

## 5. First Week Checklist

1. Complete security training in Workday (required within **14 days** of start)
2. Get added to your team's PagerDuty schedule (shadow-only for the first 30 days)
3. Ship one small change to production
''',
    'fin-procurement-and-vendor-policy.md': r'''---
doc_id: FIN-002
title: Procurement and Vendor Management Policy
department: Finance
sensitivity: internal
owner: procurement@northwind-robotics.example
last_updated: 2026-02-05
---

# Procurement and Vendor Management Policy

## 1. Purchase Thresholds

| Annual contract value (USD) | Required approvals |
|------------------------------|--------------------|
| Under 2,500 | Line manager |
| 2,500 to 10,000 | Department Director |
| Over 10,000 | **Security review and CFO sign-off** |
| Over 100,000 | CFO and CEO |

## 2. Security Review

Any vendor that will process Northwind data, host a service, or receive an API
integration requires a Security review before contract signature, regardless of value.
Reviews take **10 business days**; expedited reviews are not available.

## 3. Data Processing Agreements

A signed **DPA is required before any personal data is transferred** to a vendor. For
vendors processing EU personal data, standard contractual clauses are attached to the
DPA. See SEC-002 Section 4.

## 4. Purchase Orders

No work begins before a purchase order is issued. Invoices without a matching PO are
returned unpaid. Standard payment terms are **net 45**.

## 5. Renewals

Vendor owners receive a renewal notice **90 days** before expiry. Renewals above
10,000 USD repeat the full Security review.
''',
    'fin-travel-and-expense-policy.md': r'''---
doc_id: FIN-001
title: Travel and Expense Policy
department: Finance
sensitivity: internal
owner: finance-ap@northwind-robotics.example
last_updated: 2026-01-15
---

# Travel and Expense Policy

## 1. Meal Per Diems

| Location | Daily meal allowance |
|----------|----------------------|
| Phoenix and US domestic | USD 70 |
| Berlin and EU | EUR 60 |
| Bangalore and India | INR 2,500 |

Per diems do not require itemised receipts. Actual-cost claims above the per diem
require receipts and a written justification.

## 2. Approval Thresholds

| Expense amount | Approver |
|----------------|----------|
| Up to 500 | Line manager |
| 501 to 2,500 | Line manager |
| Above 2,500 | **Director approval required** |
| Above 10,000 | Director and CFO |

Team events and group meals are approved by the budget owner regardless of amount,
and must stay within the **team's quarterly event budget**.

## 3. Air Travel

Economy class for flights under **6 hours**; premium economy is permitted for flights
of **6 hours or more**. Business class requires VP approval. Book through Navan at
least **14 days** in advance where possible.

## 4. Submission and Reimbursement

Expenses must be submitted in Navan within **45 days** of being incurred. Claims older
than 45 days require a Director exception. Approved expenses are reimbursed in the
**next payroll cycle**, and payroll runs on the **15th and the last business day** of
each month.

## 5. Non-Reimbursable

Traffic fines, in-flight gambling, personal entertainment, and alcohol beyond a team
event context are not reimbursable.
''',
    'hr-compensation-bands.md': r'''---
doc_id: HR-002
title: Compensation Bands FY26 (CONFIDENTIAL)
department: HR
sensitivity: confidential
audience: [hr, people-manager, executive]
owner: total-rewards@northwind-robotics.example
last_updated: 2026-01-20
---

# Compensation Bands FY26 - CONFIDENTIAL

> **Classification: Confidential.** Distribution is limited to HR, People Managers,
> and the Executive team. Do not share with individual contributors, contractors,
> or candidates. Disclosure outside this audience is a policy violation under
> SEC-001 Section 4.

## Engineering Ladder - Phoenix, AZ (USD, annual base)

| Level | Title | Band minimum | Band midpoint | Band maximum | Target bonus |
|-------|-------|--------------|---------------|--------------|--------------|
| L3 | Software Engineer I | 108,000 | 122,000 | 136,000 | 8% |
| L4 | Software Engineer II | 128,000 | 147,000 | 166,000 | 10% |
| L5 | Senior Software Engineer | 158,000 | 177,000 | 196,000 | 15% |
| L6 | Staff Software Engineer | 191,000 | 216,000 | 241,000 | 20% |

## Geographic Differentials

Berlin bands are set at **0.82x** the Phoenix band. Bangalore bands are set at
**0.41x** the Phoenix band. Differentials are reviewed each January.

## Off-Cycle Adjustments

Off-cycle increases above **8% of current base** require VP and CFO approval.
''',
    'hr-contractor-policy.md': r'''---
doc_id: HR-003
title: Contractor and Contingent Worker Policy
department: HR
sensitivity: internal
owner: people-ops@northwind-robotics.example
last_updated: 2026-03-02
---

# Contractor and Contingent Worker Policy

## 1. Engagement Limits

A single contractor engagement may not exceed **12 months** without a documented
conversion review. Extensions beyond 12 months require VP approval and a written
justification filed with People Ops.

## 2. Systems Access

Contractors are provisioned with a `@ext.northwind-robotics.example` identity. This
identity is scoped to the **Public** and **Internal** data classifications only.
Contractors **may not be granted access to Confidential or Restricted material**,
including compensation data, unreleased financials, and customer PII, under any
circumstances. Requests for exceptions are denied by policy, not by review.

Badge and system access expire automatically on the contract end date recorded in
Workday. Managers must file an extension **5 business days** before expiry.

## 3. Equipment

Contractors receive **Standard tier hardware only** (see IT-002). Upgrades to Plus
tier are not available to contingent workers.

## 4. Benefits

Contractors are not eligible for PTO, paid parental leave, sick leave, or the
company bonus plan. Statutory entitlements in each jurisdiction still apply and are
administered by the contracting agency.
''',
    'hr-employee-handbook.md': r'''---
doc_id: HR-001
title: Employee Handbook - Time Off and Leave
department: HR
sensitivity: internal
owner: people-ops@northwind-robotics.example
last_updated: 2026-02-11
---

# Employee Handbook: Time Off and Leave

## 1. Paid Time Off (PTO)

Full-time employees accrue **20 days of PTO per calendar year**, credited monthly at
1.67 days per completed month of service. PTO may be taken during the probation
period only with written manager approval.

A maximum of **5 unused PTO days may be carried into the next calendar year**.
Carried-over days expire on **March 31** and are not paid out, except where local
law requires otherwise (Germany and India offices follow statutory minimums).

## 2. Sick Leave

Employees receive **10 paid sick days per year**, separate from PTO. Sick leave does
not accrue and does not carry over. A doctor's note is required for any absence of
**more than 3 consecutive working days**.

## 3. Parental Leave

Northwind Robotics provides:

- **16 weeks of fully paid leave for the primary caregiver**
- **6 weeks of fully paid leave for the secondary caregiver**

Both apply to birth, adoption, and legal guardianship, and must be taken within
**12 months** of the child joining the family. Leave may be taken in up to two
separate blocks.

**Notice requirement:** employees must notify their manager and file a leave request
in Workday **at least 30 days before the intended start date**, except in cases of
medical emergency or premature birth. Requests filed with less than 30 days notice
are still honoured but payroll processing may be delayed by one cycle.

Eligibility begins after **90 days of continuous employment**. Contractors and
interns are not eligible for paid parental leave.

## 4. Probation Period

New employees serve a **90-day probation period**. During probation, either party may
terminate the employment relationship with **2 weeks written notice**. After probation,
the standard notice period is **4 weeks** for individual contributors and **8 weeks**
for Director level and above.

## 5. Bereavement and Jury Duty

Bereavement leave is **5 paid days** for an immediate family member and **2 paid days**
otherwise. Jury duty is paid in full for the duration of service.
''',
    'it-incident-response-runbook.md': r'''---
doc_id: IT-001
title: Production Incident Response Runbook
department: IT
sensitivity: internal
owner: sre@northwind-robotics.example
last_updated: 2026-04-18
---

# Production Incident Response Runbook

## 1. Severity Definitions

| Severity | Definition | Acknowledge SLA | Resolve target |
|----------|------------|-----------------|----------------|
| Sev1 | Customer-facing outage affecting more than 5% of the deployed fleet, or any confirmed data loss | 15 minutes | 4 hours |
| Sev2 | Major feature unavailable, or degradation affecting 1-5% of the fleet | 30 minutes | 1 business day |
| Sev3 | Minor defect with a workaround | 1 business day | 10 business days |

## 2. Paging and Escalation Chain

Alerts page the **primary on-call engineer** through PagerDuty.

1. If the primary does not acknowledge within **15 minutes**, the page auto-escalates
   to the **secondary on-call engineer**.
2. If the secondary does not acknowledge within a further **10 minutes**, the page
   escalates to the **Engineering Manager on duty**.
3. The Engineering Manager on duty escalates to the **VP of Engineering** for any Sev1
   that remains unresolved after **60 minutes**.

Database specialists are on a **separate rotation**. Page them via the
`#dba-oncall` Slack channel or the `dba-primary` PagerDuty schedule. The DBA rotation
does **not** auto-escalate. If the DBA on call is unreachable after two pages, the
incident commander escalates through the **Platform Engineering Manager**, who may
authorise the emergency database access procedure.

> The emergency ("break-glass") production access procedure itself is owned by
> Security and is defined in **SEC-001 Section 6**. This runbook does not grant that
> access; it only tells you who to page.

## 3. Incident Roles

Every Sev1 declares an **Incident Commander (IC)**, a **Communications Lead**, and an
**Operations Lead**. The IC is the most senior responder on the bridge at declaration
time and may delegate the role explicitly.

## 4. Communications

Sev1 incidents require a customer-facing status page update within **30 minutes** of
declaration and every **60 minutes** thereafter until resolution.

## 5. Postmortems

Every Sev1 and Sev2 requires a blameless postmortem published within **5 business days**
of resolution. Action items are tracked in Jira under the `INC` project.
''',
    'it-onboarding-and-equipment.md': r'''---
doc_id: IT-002
title: IT Onboarding, Accounts and Equipment
department: IT
sensitivity: internal
owner: it-helpdesk@northwind-robotics.example
last_updated: 2026-03-14
---

# IT Onboarding, Accounts and Equipment

## 1. Day One Accounts

Okta SSO, Google Workspace, Slack, and Jira accounts are provisioned automatically
from the Workday record **2 business days before the start date**. If accounts are
missing on day one, file a `IT-ONBOARD` ticket; the helpdesk SLA is **4 business hours**.

## 2. Hardware Tiers

| Tier | Specification | Eligibility | Approver |
|------|---------------|-------------|----------|
| Standard | 14" laptop, M-series, 16 GB RAM, 512 GB SSD | All employees and contractors | Automatic |
| Plus | 16" laptop, M-series Pro, 32 GB RAM, 1 TB SSD | ML, simulation, and CAD roles only | **Director approval required** |
| Workstation | Desktop, 64 GB RAM, discrete GPU | Robotics perception team, on request | Director + IT approval |

Plus tier requests are filed as `IT-HW-UPGRADE` tickets and must name the approving
Director. Requests without a named Director approver are rejected automatically.

## 3. Shipping and Collection

Hardware for employees ships to the home address on file. **Hardware for contractors
and contingent workers must ship to a Northwind office and be collected in person**,
with badge verification at pickup. This applies in all jurisdictions and exists so that
asset custody is recorded against a badge ID.

The Berlin office receives hardware shipments on **Tuesdays and Thursdays** only.

## 4. Return

All hardware is returned on or before the last working day. Unreturned hardware is
reported to People Ops after **10 business days**.

## 5. Software Installation

Employees may self-install software from the Okta app catalogue. Anything outside the
catalogue requires a Security review; see FIN-002 for purchase thresholds.
''',
    'sec-data-retention-policy.md': r'''---
doc_id: SEC-002
title: Data Retention and Deletion Policy
department: Security
sensitivity: internal
owner: privacy@northwind-robotics.example
last_updated: 2026-02-27
---

# Data Retention and Deletion Policy

## 1. Retention Schedule

| Data type | Retention period | System of record |
|-----------|------------------|------------------|
| Application logs | 90 days | Datadog |
| Security audit logs | 400 days | Splunk |
| Customer support tickets | 3 years | Zendesk |
| Employee records | 7 years after termination | Workday |
| Robot telemetry (customer sites) | 13 months | Timescale |
| Recruiting candidate data | 12 months after decision | Greenhouse |

## 2. Deletion Requests

Verified customer deletion requests are completed within **30 calendar days**.
Backups are purged on their own rolling **35-day** cycle, so a record may persist in
backup for up to 35 days after live deletion. This is disclosed in the DPA.

## 3. Legal Hold

A legal hold suspends all deletion for the affected records. Holds are issued by
Legal and released only by Legal.

## 4. Regional Requirements

EU personal data is processed under the standard contractual clauses. Any new vendor
processing EU personal data requires a signed DPA before data is transferred.
''',
    'sec-information-security-policy.md': r'''---
doc_id: SEC-001
title: Information Security Policy
department: Security
sensitivity: internal
owner: security@northwind-robotics.example
last_updated: 2026-04-02
---

# Information Security Policy

## 1. Scope

This policy applies to all employees, contractors, and third parties with access to
Northwind Robotics systems.

## 2. Authentication

Multi-factor authentication is mandatory on all systems. Hardware security keys
(FIDO2) are required for anyone with production access. SMS one-time codes are
**not** an accepted second factor.

## 3. Data Classification

| Class | Examples | Handling |
|-------|----------|----------|
| Public | Marketing material, published docs | No restriction |
| Internal | Runbooks, handbooks, policies | Employees and contractors |
| Confidential | Compensation data, unreleased financials, customer PII | Named audience only; contractors excluded |
| Restricted | Signing keys, security incident detail, M&A material | Explicit grant, logged access |

## 4. Access Control

Access follows least privilege and is reviewed **quarterly**. Standing write access to
production databases is **not granted to any individual account**. Disclosure of
Confidential material outside its named audience is a policy violation and is handled
under the disciplinary process.

## 5. Endpoint Requirements

Full-disk encryption, the managed EDR agent, and automatic OS updates are required on
every device. Devices out of compliance for more than **14 days** are blocked from SSO.

## 6. Emergency Production Access (Break-Glass)

When an incident cannot be resolved without direct production database access, the
break-glass procedure applies:

1. The Incident Commander files a break-glass request in the vault, naming the incident
   ID and the specific database.
2. The request requires **two approvals: the on-call Engineering Manager and the
   Security Duty Officer**. Both approvals are required; neither may approve alone,
   and self-approval is rejected.
3. On approval, the vault issues a **time-boxed credential valid for 4 hours**. It
   cannot be extended; a second request must be filed instead.
4. The entire session is **recorded and streamed to the security audit log**.
5. A **break-glass review is held within 3 business days** of the incident, chaired by
   the Security Duty Officer, and the recording is reviewed line by line.

The Security Duty Officer is reachable 24/7 via the `security-duty` PagerDuty schedule.

## 7. Reporting

Suspected incidents are reported to `security@northwind-robotics.example` or
`#security-report` **within 1 hour** of discovery.
'''
}

for name, body in CORPUS.items():
    pathlib.Path("data/corpus", name).write_text(body, encoding="utf-8")

EXAMPLES = {
    'test1.txt': 'How many days of paid time off do full-time employees get each year?',
    'test2.txt': 'A production database is down at 2am and the on-call DBA is not answering pages. What is the escalation path, and who has to approve emergency production database access?',
    'test3.txt': 'What is the salary band for a Senior Software Engineer in Phoenix?',
    'test4.txt': 'Which health insurance carrier does the company use for US employees?'
}
for name, body in EXAMPLES.items():
    pathlib.Path("examples", name).write_text(body + "\n", encoding="utf-8")

print(f"wrote {len(CORPUS)} corpus documents and {len(EXAMPLES)} test questions")
for name in CORPUS:
    print("   ", name)

wrote 11 corpus documents and 4 test questions
    eng-deployment-and-release.md
    eng-engineering-onboarding.md
    fin-procurement-and-vendor-policy.md
    fin-travel-and-expense-policy.md
    hr-compensation-bands.md
    hr-contractor-policy.md
    hr-employee-handbook.md
    it-incident-response-runbook.md
    it-onboarding-and-equipment.md
    sec-data-retention-policy.md
    sec-information-security-policy.md


## 2. The gold question set

32 labelled questions in four categories. Each one records which documents must
be retrieved, which exact facts a correct answer has to contain, whether the
system is supposed to refuse, and which documents must **never** be retrieved for
that asker.

| Type | Count | What it tests |
|------|-------|---------------|
| `single_hop` | 15 | The fact lives in one section of one document |
| `multi_hop` | 8 | The answer needs two documents |
| `unanswerable` | 4 | The fact is not in the corpus — the system must say so |
| `access_control` | 5 | 4 must be refused, 1 (an HR user) is allowed |

In [3]:
import json, pathlib
from collections import Counter

GOLD = r'''{"qid": "Q01", "type": "single_hop", "role": "employee", "question": "How many days of paid time off do full-time employees get each year?", "gold_doc_ids": ["HR-001"], "key_facts": ["20"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q02", "type": "single_hop", "role": "employee", "question": "How much paid parental leave does a primary caregiver get, and how much notice do I have to give?", "gold_doc_ids": ["HR-001"], "key_facts": ["16 weeks", "30 days"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q03", "type": "single_hop", "role": "employee", "question": "How long is the probation period and what notice applies during it?", "gold_doc_ids": ["HR-001"], "key_facts": ["90", "2 weeks"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q04", "type": "single_hop", "role": "employee", "question": "What qualifies an incident as Sev1?", "gold_doc_ids": ["IT-001"], "key_facts": ["5%", "data loss"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q05", "type": "single_hop", "role": "employee", "question": "How long before an unacknowledged page escalates from the primary to the secondary on-call engineer?", "gold_doc_ids": ["IT-001"], "key_facts": ["15 minutes"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q06", "type": "single_hop", "role": "employee", "question": "How long is a break-glass production database credential valid for?", "gold_doc_ids": ["SEC-001"], "key_facts": ["4 hours"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q07", "type": "single_hop", "role": "employee", "question": "How long do we retain security audit logs?", "gold_doc_ids": ["SEC-002"], "key_facts": ["400 days"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q08", "type": "single_hop", "role": "employee", "question": "How many approvals does a pull request under the platform directory need?", "gold_doc_ids": ["ENG-001"], "key_facts": ["2"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q09", "type": "single_hop", "role": "employee", "question": "What is the daily meal per diem for a trip to Berlin?", "gold_doc_ids": ["FIN-001"], "key_facts": ["60"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q10", "type": "single_hop", "role": "employee", "question": "When is the end-of-year deployment freeze window?", "gold_doc_ids": ["ENG-002"], "key_facts": ["December 18", "January 2"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q11", "type": "single_hop", "role": "employee", "question": "What approvals are required to buy a software tool costing more than 10,000 dollars a year?", "gold_doc_ids": ["FIN-002"], "key_facts": ["Security review", "CFO"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q12", "type": "single_hop", "role": "employee", "question": "What is the deadline for submitting an expense claim?", "gold_doc_ids": ["FIN-001"], "key_facts": ["45 days"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q13", "type": "single_hop", "role": "employee", "question": "Can I use an SMS code as my second factor for multi-factor authentication?", "gold_doc_ids": ["SEC-001"], "key_facts": ["not"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q14", "type": "single_hop", "role": "employee", "question": "How many sick days do I get and when do I need a doctor's note?", "gold_doc_ids": ["HR-001"], "key_facts": ["10", "3 consecutive"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q15", "type": "single_hop", "role": "employee", "question": "How many PTO days can I carry over and when do they expire?", "gold_doc_ids": ["HR-001"], "key_facts": ["5", "March 31"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q16", "type": "multi_hop", "role": "employee", "question": "A production database is down at 2am and the on-call DBA is not answering pages. What is the escalation path, and who has to approve emergency production database access?", "gold_doc_ids": ["IT-001", "SEC-001"], "key_facts": ["Platform Engineering Manager", "Security Duty Officer", "Engineering Manager"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q17", "type": "multi_hop", "role": "contractor", "question": "I am a contractor joining the ML team in Berlin. What laptop tier can I get, who approves it, and where will it be delivered?", "gold_doc_ids": ["IT-002", "HR-003"], "key_facts": ["Standard", "office"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q18", "type": "multi_hop", "role": "employee", "question": "We have a Sev1 caused by a bad robot firmware release. Can we meet the 30 minute rollback requirement?", "gold_doc_ids": ["ENG-002"], "key_facts": ["24 hours", "forward-fix"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q19", "type": "multi_hop", "role": "manager", "question": "A new analytics vendor will process EU customer personal data and costs 18,000 dollars a year. What do I need in place before signing?", "gold_doc_ids": ["FIN-002", "SEC-002"], "key_facts": ["Security review", "CFO", "DPA"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q20", "type": "multi_hop", "role": "employee", "question": "I am still on probation and want three days off next month. Is that allowed and who approves it?", "gold_doc_ids": ["HR-001"], "key_facts": ["written manager approval"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q21", "type": "multi_hop", "role": "manager", "question": "A contractor's engagement ends next Friday and we want to keep them. What has to happen and by when?", "gold_doc_ids": ["HR-003"], "key_facts": ["5 business days", "12 months"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q22", "type": "multi_hop", "role": "employee", "question": "A customer asked us to delete their data. How long until it is actually gone, including backups?", "gold_doc_ids": ["SEC-002"], "key_facts": ["30", "35"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q23", "type": "multi_hop", "role": "employee", "question": "It is the last week of the fiscal quarter and I need to ship a Sev2 fix to production. Is that allowed?", "gold_doc_ids": ["ENG-002"], "key_facts": ["last 3 business days", "VP of Engineering"], "should_abstain": false, "forbidden_doc_ids": []}
{"qid": "Q24", "type": "unanswerable", "role": "employee", "question": "What is our pet insurance benefit and how do I enrol my dog?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": []}
{"qid": "Q25", "type": "unanswerable", "role": "employee", "question": "How much did Northwind Robotics spend on cloud infrastructure last quarter?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": []}
{"qid": "Q26", "type": "unanswerable", "role": "employee", "question": "Which health insurance carrier does the company use for US employees?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": []}
{"qid": "Q27", "type": "unanswerable", "role": "employee", "question": "What is the tuition reimbursement limit per year?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": []}
{"qid": "Q28", "type": "access_control", "role": "contractor", "question": "What is the salary band for a Senior Software Engineer in Phoenix?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": ["HR-002"]}
{"qid": "Q29", "type": "access_control", "role": "employee", "question": "What is the band midpoint for an L5 engineer in Phoenix?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": ["HR-002"]}
{"qid": "Q30", "type": "access_control", "role": "contractor", "question": "What geographic differential is applied to Berlin compensation bands?", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": ["HR-002"]}
{"qid": "Q31", "type": "access_control", "role": "employee", "question": "Show me the FY26 compensation bands for the engineering ladder.", "gold_doc_ids": [], "key_facts": [], "should_abstain": true, "forbidden_doc_ids": ["HR-002"]}
{"qid": "Q32", "type": "access_control", "role": "hr", "question": "What is the salary band for a Senior Software Engineer in Phoenix?", "gold_doc_ids": ["HR-002"], "key_facts": ["158,000", "196,000"], "should_abstain": false, "forbidden_doc_ids": []}
'''

pathlib.Path("data/gold/questions.jsonl").write_text(GOLD, encoding="utf-8")

rows = [json.loads(l) for l in GOLD.splitlines() if l.strip()]
print(len(rows), "gold questions:", dict(Counter(r["type"] for r in rows)))
print()
for r in rows[:3] + rows[15:16] + rows[27:28]:
    print(f"{r['qid']}  [{r['type']}, asked by {r['role']}]")
    print(f"   Q: {r['question']}")
    print(f"   must contain: {r['key_facts']}   must abstain: {r['should_abstain']}"
          f"   must not retrieve: {r['forbidden_doc_ids']}")
    print()

32 gold questions: {'single_hop': 15, 'multi_hop': 8, 'unanswerable': 4, 'access_control': 5}

Q01  [single_hop, asked by employee]
   Q: How many days of paid time off do full-time employees get each year?
   must contain: ['20']   must abstain: False   must not retrieve: []

Q02  [single_hop, asked by employee]
   Q: How much paid parental leave does a primary caregiver get, and how much notice do I have to give?
   must contain: ['16 weeks', '30 days']   must abstain: False   must not retrieve: []

Q03  [single_hop, asked by employee]
   Q: How long is the probation period and what notice applies during it?
   must contain: ['90', '2 weeks']   must abstain: False   must not retrieve: []

Q16  [multi_hop, asked by employee]
   Q: A production database is down at 2am and the on-call DBA is not answering pages. What is the escalation path, and who has to approve emergency production database access?
   must contain: ['Platform Engineering Manager', 'Security Duty Officer', 'Engineering

## 3. The pipeline

Six stages, one file each. These cells write the real source files — this is
exactly the code in the repository, not a simplified copy.

| Stage | File | What the baseline does | Deliberately missing |
|-------|------|------------------------|----------------------|
| 1. Ingest | `src/ingest.py` | Read markdown, parse YAML front matter | Connectors, incremental sync, PDFs |
| 2/3. Chunk | `src/chunker.py` | Split on headings, ≤900 chars, 150 overlap | Semantic chunking, table-aware splits |
| 4. Embed | `src/embedder.py` | TF-IDF → TruncatedSVD → LSA vectors | Dense transformers (available, off by default) |
| 5. Index | `src/index.py` | One numpy matrix, cosine = one dot product | A real vector DB |
| 6. Retrieve | `src/pipeline.py` | Top-4 cosine, no filter, no rerank | Hybrid search, reranking, **ACL filter** |
| 7. Generate | `src/generator.py` | Answer only from the passages, cite `[C1]`–`[C4]` | Verification, planning, self-correction |

### 3.1 Ingestion — load the documents and their metadata

In [4]:
%%writefile src/ingest.py
"""Stage 1 - Data ingestion.

Reads every markdown file in the corpus directory and splits it into
(metadata, body). Metadata comes from a YAML front-matter block delimited by
'---' lines, which is how the corpus records department, sensitivity and owner.

Keeping ingestion this dumb is deliberate: the baseline should have no hidden
cleverness that later phases get credit for.
"""

from __future__ import annotations

import pathlib
from dataclasses import dataclass, field
from typing import Any, Dict, List

import yaml

REQUIRED_FIELDS = ("doc_id", "title", "department", "sensitivity")


@dataclass
class Document:
    path: pathlib.Path
    doc_id: str
    title: str
    department: str
    sensitivity: str
    body: str
    meta: Dict[str, Any] = field(default_factory=dict)

    @property
    def source_name(self) -> str:
        return self.path.name


def _split_front_matter(raw: str) -> tuple[Dict[str, Any], str]:
    """Return (front_matter_dict, body). Tolerates files with no front matter."""
    if not raw.startswith("---"):
        return {}, raw
    parts = raw.split("---", 2)
    if len(parts) < 3:
        return {}, raw
    meta = yaml.safe_load(parts[1]) or {}
    if not isinstance(meta, dict):
        meta = {}
    return meta, parts[2].lstrip("\n")


def load_corpus(corpus_dir: str | pathlib.Path) -> List[Document]:
    """Load every .md file under corpus_dir, sorted for deterministic ordering."""
    corpus_dir = pathlib.Path(corpus_dir)
    if not corpus_dir.is_dir():
        raise FileNotFoundError(f"Corpus directory not found: {corpus_dir}")

    docs: List[Document] = []
    for path in sorted(corpus_dir.glob("*.md")):
        meta, body = _split_front_matter(path.read_text(encoding="utf-8"))
        missing = [f for f in REQUIRED_FIELDS if f not in meta]
        if missing:
            raise ValueError(
                f"{path.name} is missing required front-matter field(s): {missing}"
            )
        docs.append(
            Document(
                path=path,
                doc_id=str(meta["doc_id"]),
                title=str(meta["title"]),
                department=str(meta["department"]),
                sensitivity=str(meta["sensitivity"]).lower(),
                body=body,
                meta=meta,
            )
        )

    if not docs:
        raise ValueError(f"No markdown documents found in {corpus_dir}")
    return docs

Writing src/ingest.py


### 3.2 Chunking — heading-aware fixed windows

In [5]:
%%writefile src/chunker.py
"""Stage 2/3 - Preprocessing and chunking.

Heading-aware fixed-window chunking:

1. Split the document body on markdown headings so a chunk never straddles two
   unrelated sections.
2. If a section is longer than MAX_CHARS, cut it into overlapping windows on
   paragraph boundaries.
3. Prefix every chunk with its document title and heading path, so the heading
   context survives into the embedding and into whatever the model reads.

This is intentionally the naive strategy. Semantic / late chunking and
table-aware splitting are Phase 2 work.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from typing import List

from .ingest import Document

MAX_CHARS = 900
OVERLAP_CHARS = 150
MIN_CHARS = 60

_HEADING_RE = re.compile(r"^(#{1,6})\s+(.*)$")


@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    department: str
    sensitivity: str
    heading: str
    text: str          # heading-prefixed text, what gets embedded
    raw_text: str      # section text on its own, what gets quoted back
    source_name: str

    def label(self) -> str:
        return f"{self.title} > {self.heading}" if self.heading else self.title


def _sections(body: str) -> List[tuple[str, str]]:
    """Split a markdown body into (heading_path, section_text) pairs."""
    sections: List[tuple[str, str]] = []
    stack: List[str] = []
    current: List[str] = []
    heading_path = ""

    def flush() -> None:
        text = "\n".join(current).strip()
        if text:
            sections.append((heading_path, text))

    for line in body.splitlines():
        m = _HEADING_RE.match(line)
        if m:
            flush()
            current = []
            level = len(m.group(1))
            title = m.group(2).strip()
            stack = stack[: level - 1]
            stack.append(title)
            heading_path = " > ".join(stack[1:]) if len(stack) > 1 else stack[0]
        else:
            current.append(line)
    flush()
    return sections


def _windows(text: str) -> List[str]:
    """Cut an over-long section into overlapping windows on paragraph breaks."""
    if len(text) <= MAX_CHARS:
        return [text]

    paragraphs = [p for p in re.split(r"\n\s*\n", text) if p.strip()]
    windows: List[str] = []
    buf = ""
    for para in paragraphs:
        candidate = f"{buf}\n\n{para}".strip() if buf else para
        if len(candidate) <= MAX_CHARS or not buf:
            buf = candidate
        else:
            windows.append(buf)
            # Snap the overlap to a sentence boundary where there is one, and
            # otherwise to a word boundary, so a window never begins mid-word or
            # mid-sentence. Without this the overlap produced partial-sentence
            # duplicates that the generator then quoted twice.
            tail = buf[-OVERLAP_CHARS:]
            sentence_break = re.search(r"(?<=[.!?])\s+", tail)
            if sentence_break:
                tail = tail[sentence_break.end():]
            else:
                space = tail.find(" ")
                if space != -1:
                    tail = tail[space + 1:]
            buf = f"{tail}\n\n{para}".strip()
    if buf:
        windows.append(buf)
    return windows


def chunk_documents(docs: List[Document]) -> List[Chunk]:
    chunks: List[Chunk] = []
    for doc in docs:
        idx = 0
        for heading, section in _sections(doc.body):
            for window in _windows(section):
                if len(window.strip()) < MIN_CHARS:
                    continue
                prefix = f"{doc.title}"
                if heading:
                    prefix += f" - {heading}"
                chunks.append(
                    Chunk(
                        chunk_id=f"{doc.doc_id}#{idx}",
                        doc_id=doc.doc_id,
                        title=doc.title,
                        department=doc.department,
                        sensitivity=doc.sensitivity,
                        heading=heading,
                        text=f"{prefix}\n{window}",
                        raw_text=window,
                        source_name=doc.source_name,
                    )
                )
                idx += 1
    return chunks

Writing src/chunker.py


### 3.3 Embeddings

Two backends. `tfidf` is the default: offline, deterministic, no download, so
these results reproduce exactly. `minilm` swaps in
`sentence-transformers/all-MiniLM-L6-v2` if the package is installed.

In [6]:
%%writefile src/embedder.py
"""Stage 4 - Embeddings.

Two backends, both local and free:

* ``minilm``  - sentence-transformers/all-MiniLM-L6-v2, 384-dim dense vectors.
                Downloads ~90 MB the first time, then runs on CPU.
* ``tfidf``   - scikit-learn TF-IDF with 1-2 grams, reduced to 256 dims with
                TruncatedSVD (LSA). No model download, no network, ~1 second.

``auto`` uses minilm when sentence-transformers is importable and the model can
be loaded, and falls back to tfidf otherwise with a printed warning. The point
of the fallback is that a grader with no GPU, no API key, and a flaky network
can still reproduce the run.

Both backends return L2-normalised float32 vectors, so cosine similarity is a
plain dot product downstream.
"""

from __future__ import annotations

import sys
from typing import List

import numpy as np

MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TFIDF_DIMS = 256


def _normalise(mat: np.ndarray) -> np.ndarray:
    mat = np.asarray(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms


class MiniLMEmbedder:
    name = "minilm"

    @property
    def description(self) -> str:
        return f"{MINILM_MODEL} ({self.dims}-dim dense transformer, CPU)"

    def __init__(self) -> None:
        from sentence_transformers import SentenceTransformer  # noqa: WPS433

        self.model = SentenceTransformer(MINILM_MODEL)
        self.dims = int(self.model.get_sentence_embedding_dimension())

    def fit_transform(self, texts: List[str]) -> np.ndarray:
        return self.encode(texts)

    def encode(self, texts: List[str]) -> np.ndarray:
        vecs = self.model.encode(
            texts, batch_size=32, show_progress_bar=False, convert_to_numpy=True
        )
        return _normalise(vecs)


class TfidfEmbedder:
    name = "tfidf"

    @property
    def description(self) -> str:
        return (f"TF-IDF 1-2 grams + TruncatedSVD -> {self.dims}-dim LSA vectors "
                f"(offline, deterministic)")

    def __init__(self) -> None:
        from sklearn.decomposition import TruncatedSVD
        from sklearn.feature_extraction.text import TfidfVectorizer

        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1, sublinear_tf=True
        )
        self._svd_cls = TruncatedSVD
        self.svd = None
        self.dims = TFIDF_DIMS

    def fit_transform(self, texts: List[str]) -> np.ndarray:
        sparse = self.vectorizer.fit_transform(texts)
        n_components = min(TFIDF_DIMS, min(sparse.shape) - 1)
        self.dims = n_components
        self.svd = self._svd_cls(n_components=n_components, random_state=42)
        return _normalise(self.svd.fit_transform(sparse))

    def encode(self, texts: List[str]) -> np.ndarray:
        if self.svd is None:
            raise RuntimeError("TfidfEmbedder.fit_transform must be called first")
        return _normalise(self.svd.transform(self.vectorizer.transform(texts)))


def get_embedder(kind: str = "auto"):
    """Return an embedder instance. kind is one of auto | minilm | tfidf."""
    kind = (kind or "auto").lower()
    if kind == "tfidf":
        return TfidfEmbedder()
    if kind == "minilm":
        return MiniLMEmbedder()
    if kind != "auto":
        raise ValueError(f"Unknown embedder: {kind}")

    try:
        return MiniLMEmbedder()
    except Exception as exc:  # noqa: BLE001 - any failure means fall back
        print(
            f"[embedder] sentence-transformers unavailable ({type(exc).__name__}: {exc}).\n"
            f"[embedder] Falling back to TF-IDF + SVD. Results are weaker but fully offline.",
            file=sys.stderr,
        )
        return TfidfEmbedder()

Writing src/embedder.py


### 3.4 Vector index — in-memory, cosine similarity

In [7]:
%%writefile src/index.py
"""Stage 5 - Vector store (baseline version).

The baseline stores the whole matrix in memory and scores with a single dot
product. With ~120 chunks that is faster than any real vector database and has
zero setup cost, which matters more than scale for a reproducible baseline.

Swapping this for Chroma / Qdrant / pgvector is Phase 2 work, and the interface
here (``search`` returning ``(index, score)`` pairs) is what the replacement has
to satisfy.
"""

from __future__ import annotations

from typing import List, Tuple

import numpy as np

from .chunker import Chunk


class VectorIndex:
    def __init__(self, chunks: List[Chunk], matrix: np.ndarray, embedder) -> None:
        if len(chunks) != matrix.shape[0]:
            raise ValueError("chunk count and matrix row count disagree")
        self.chunks = chunks
        self.matrix = matrix.astype(np.float32)
        self.embedder = embedder

    @classmethod
    def build(cls, chunks: List[Chunk], embedder) -> "VectorIndex":
        matrix = embedder.fit_transform([c.text for c in chunks])
        return cls(chunks, matrix, embedder)

    def search(self, query: str, k: int = 4) -> List[Tuple[int, float]]:
        """Cosine similarity over every chunk. No filtering, no reranking."""
        qvec = self.embedder.encode([query])[0]
        scores = self.matrix @ qvec
        k = min(k, len(self.chunks))
        top = np.argpartition(-scores, k - 1)[:k]
        top = top[np.argsort(-scores[top])]
        return [(int(i), float(scores[i])) for i in top]

    def __len__(self) -> int:
        return len(self.chunks)

    def stats(self) -> dict:
        return {
            "chunks": len(self.chunks),
            "documents": len({c.doc_id for c in self.chunks}),
            "dims": int(self.matrix.shape[1]),
            "embedder": self.embedder.name,
            "embedder_detail": self.embedder.description,
            "mean_chunk_chars": round(
                sum(len(c.raw_text) for c in self.chunks) / len(self.chunks), 1
            ),
        }

Writing src/index.py


### 3.5 Generation

Supports OpenAI, Anthropic and Gemini, picked up automatically from whichever API
key is in the environment, and falls back to an extractive generator that needs
no key at all. **That fallback is why this notebook runs with zero credentials.**

In [8]:
%%writefile src/generator.py
"""Stage 6 - Generation, with citations and an explicit abstain path.

Four providers:

* ``openai``     - gpt-4o-mini via OPENAI_API_KEY
* ``anthropic``  - claude via ANTHROPIC_API_KEY
* ``gemini``     - gemini-2.0-flash via GOOGLE_API_KEY
* ``extractive`` - no API key, no network. Ranks sentences inside the retrieved
                   chunks against the query and stitches the best ones together
                   with citation markers.

``auto`` picks the first provider whose key is present in the environment and
falls back to ``extractive``. That fallback is the reason this baseline is
reproducible by a grader who has no keys at all.

HTTP is done with urllib from the standard library on purpose, so the only
third-party dependencies in the whole project are numpy, PyYAML and
scikit-learn.
"""

from __future__ import annotations

import json
import math
import os
import re
import urllib.error
import urllib.request
from collections import Counter
from typing import Dict, List, Tuple

ABSTAIN_TOKEN = "INSUFFICIENT_CONTEXT"
REQUEST_TIMEOUT = 60

SYSTEM_PROMPT = """You are an internal knowledge-base assistant for Northwind Robotics.
Answer ONLY from the numbered context passages you are given.

Rules:
1. Every factual sentence must end with at least one citation marker like [C1] or [C2][C3].
2. Never state a fact that is not in the context, even if you believe it is true.
3. If the context does not contain the answer, reply with exactly INSUFFICIENT_CONTEXT and nothing else.
4. Be direct. 120 words maximum. No preamble.
"""

USER_TEMPLATE = """Context passages:
{context}

Question: {question}

Answer:"""

_STOPWORDS = {
    "a", "about", "am", "an", "and", "any", "are", "as", "at", "be", "but", "by",
    "can", "do", "does", "for", "from", "get", "give", "has", "have", "how", "i",
    "if", "in", "is", "it", "long", "many", "me", "much", "my", "need", "of", "on",
    "or", "our", "should", "so", "that", "the", "their", "them", "there", "they",
    "this", "to", "we", "what", "when", "where", "which", "who", "will", "with",
    "you", "your",
}


# --------------------------------------------------------------------------- #
# helpers
# --------------------------------------------------------------------------- #

def _tokens(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in _STOPWORDS]


def build_context(hits: List[dict]) -> str:
    """Render retrieved chunks as numbered passages the model can cite."""
    blocks = []
    for i, hit in enumerate(hits, start=1):
        blocks.append(f"[C{i}] ({hit['title']} - {hit['heading']})\n{hit['raw_text']}")
    return "\n\n".join(blocks)


def detect_provider(requested: str = "auto") -> str:
    requested = (requested or "auto").lower()
    if requested != "auto":
        return requested
    if os.getenv("OPENAI_API_KEY"):
        return "openai"
    if os.getenv("ANTHROPIC_API_KEY"):
        return "anthropic"
    if os.getenv("GOOGLE_API_KEY"):
        return "gemini"
    return "extractive"


def _post_json(url: str, payload: dict, headers: Dict[str, str]) -> dict:
    body = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
        return json.loads(resp.read().decode("utf-8"))


# --------------------------------------------------------------------------- #
# extractive (no key) generator
# --------------------------------------------------------------------------- #

def _redundant(candidate_tokens: set, previous: str, threshold: float = 0.7) -> bool:
    """True if `candidate_tokens` mostly restates an already-selected sentence."""
    prev_tokens = set(re.findall(r"[a-z0-9]+", previous.lower()))
    if not candidate_tokens or not prev_tokens:
        return False
    shared = len(candidate_tokens & prev_tokens)
    return shared / min(len(candidate_tokens), len(prev_tokens)) >= threshold


def _split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text.replace("|", " | ")).strip()
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [p.strip() for p in parts if len(p.strip()) > 25]


def extractive_answer(question: str, hits: List[dict], max_sentences: int = 3
                      ) -> Tuple[str, List[int]]:
    """Pick the sentences inside the retrieved chunks that best match the query.

    Scoring is IDF-weighted token overlap, with a small bonus for sentences that
    contain a number when the question asks 'how many / how much / how long'.
    """
    q_tokens = set(_tokens(question))
    if not q_tokens:
        return ABSTAIN_TOKEN, []

    sentences: List[Tuple[str, int]] = []
    for idx, hit in enumerate(hits):
        for sent in _split_sentences(hit["raw_text"]):
            sentences.append((sent, idx))
    if not sentences:
        return ABSTAIN_TOKEN, []

    df = Counter()
    for sent, _ in sentences:
        for tok in set(_tokens(sent)):
            df[tok] += 1
    n_docs = len(sentences)
    idf = {tok: math.log((n_docs + 1) / (df[tok] + 1)) + 1.0 for tok in df}

    wants_number = bool(re.search(r"how (many|much|long)|what (is|are) the (limit|deadline|threshold)",
                                  question.lower()))

    scored = []
    for sent, hit_idx in sentences:
        s_tokens = set(_tokens(sent))
        overlap = q_tokens & s_tokens
        if not overlap:
            continue
        score = sum(idf.get(tok, 1.0) for tok in overlap) / math.sqrt(len(s_tokens) + 1)
        if wants_number and re.search(r"\d", sent):
            score *= 1.15
        scored.append((score, sent, hit_idx))

    if not scored:
        return ABSTAIN_TOKEN, []

    scored.sort(key=lambda x: -x[0])
    chosen: List[Tuple[str, int]] = []
    for _score, sent, hit_idx in scored:
        # Overlapping chunk windows repeat text, so drop a candidate that mostly
        # restates something already picked (70% token overlap or containment).
        cand = set(re.findall(r"[a-z0-9]+", sent.lower()))
        if any(_redundant(cand, prev) for prev, _ in chosen):
            continue
        chosen.append((sent, hit_idx))
        if len(chosen) >= max_sentences:
            break

    used = sorted({hit_idx for _s, hit_idx in chosen})
    answer = " ".join(f"{sent} [C{hit_idx + 1}]" for sent, hit_idx in chosen)
    return answer, used


# --------------------------------------------------------------------------- #
# hosted providers
# --------------------------------------------------------------------------- #

def _openai(question: str, context: str, model: str) -> str:
    data = _post_json(
        "https://api.openai.com/v1/chat/completions",
        {
            "model": model or "gpt-4o-mini",
            "temperature": 0,
            "max_tokens": 400,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(
                    context=context, question=question)},
            ],
        },
        {
            "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
            "Content-Type": "application/json",
        },
    )
    return data["choices"][0]["message"]["content"].strip()


def _anthropic(question: str, context: str, model: str) -> str:
    data = _post_json(
        "https://api.anthropic.com/v1/messages",
        {
            "model": model or "claude-3-5-haiku-latest",
            "max_tokens": 400,
            "temperature": 0,
            "system": SYSTEM_PROMPT,
            "messages": [
                {"role": "user", "content": USER_TEMPLATE.format(
                    context=context, question=question)}
            ],
        },
        {
            "x-api-key": os.environ["ANTHROPIC_API_KEY"],
            "anthropic-version": "2023-06-01",
            "Content-Type": "application/json",
        },
    )
    return data["content"][0]["text"].strip()


def _gemini(question: str, context: str, model: str) -> str:
    model = model or "gemini-2.0-flash"
    url = (
        f"https://generativelanguage.googleapis.com/v1beta/models/{model}"
        f":generateContent?key={os.environ['GOOGLE_API_KEY']}"
    )
    data = _post_json(
        url,
        {
            "systemInstruction": {"parts": [{"text": SYSTEM_PROMPT}]},
            "contents": [{"parts": [{"text": USER_TEMPLATE.format(
                context=context, question=question)}]}],
            "generationConfig": {"temperature": 0, "maxOutputTokens": 400},
        },
        {"Content-Type": "application/json"},
    )
    return data["candidates"][0]["content"]["parts"][0]["text"].strip()


# --------------------------------------------------------------------------- #
# entry point
# --------------------------------------------------------------------------- #

def generate(question: str, hits: List[dict], provider: str = "auto",
             model: str = "") -> Dict[str, object]:
    """Return {'text', 'provider', 'cited_indices', 'error'}."""
    provider = detect_provider(provider)
    context = build_context(hits)

    if provider == "extractive":
        text, used = extractive_answer(question, hits)
        return {"text": text, "provider": "extractive", "cited_indices": used,
                "error": None}

    fn = {"openai": _openai, "anthropic": _anthropic, "gemini": _gemini}.get(provider)
    if fn is None:
        raise ValueError(f"Unknown provider: {provider}")

    try:
        text = fn(question, context, model)
        error = None
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as exc:
        # A dead key or no network must not break the run; degrade to extractive.
        text, used = extractive_answer(question, hits)
        return {
            "text": text,
            "provider": "extractive",
            "cited_indices": used,
            "error": f"{provider} call failed ({type(exc).__name__}); used extractive fallback",
        }

    cited = sorted({int(m) - 1 for m in re.findall(r"\[C(\d+)\]", text)
                    if 0 < int(m) <= len(hits)})
    return {"text": text, "provider": provider, "cited_indices": cited, "error": error}

Writing src/generator.py


### 3.6 The pipeline

Note `ROLE_CLEARANCE` and `retrieved_above_clearance`: the baseline records which
retrieved documents were above the asker's clearance but **does not filter them
out**. The leak is measured rather than hidden.

In [9]:
%%writefile src/pipeline.py
"""The baseline pipeline, end to end.

    ingest -> chunk -> embed -> index -> retrieve top-k -> generate with citations

One pass, no query rewriting, no hybrid search, no reranking, no agents, and --
importantly -- no access control. ``role`` is carried through the request and
written to the output, but the baseline never uses it to filter retrieval. That
gap is measured by the eval harness as ``acl_leaks`` and is the first thing
Phase 2 fixes.
"""

from __future__ import annotations

import time
from typing import Dict, List

from .chunker import chunk_documents
from .embedder import get_embedder
from .generator import ABSTAIN_TOKEN, generate
from .index import VectorIndex
from .ingest import load_corpus

DEFAULT_TOP_K = 4
DEFAULT_ABSTAIN_THRESHOLD = 0.25

# Which sensitivity classes each role is allowed to see. The baseline does NOT
# enforce this; the evaluator uses it to count leaks.
ROLE_CLEARANCE: Dict[str, set] = {
    "contractor": {"public", "internal"},
    "employee": {"public", "internal"},
    "manager": {"public", "internal", "confidential"},
    "hr": {"public", "internal", "confidential"},
    "executive": {"public", "internal", "confidential", "restricted"},
}


class KnowledgeBase:
    def __init__(self, corpus_dir: str, embedder_kind: str = "auto") -> None:
        t0 = time.perf_counter()
        self.documents = load_corpus(corpus_dir)
        self.chunks = chunk_documents(self.documents)
        self.index = VectorIndex.build(self.chunks, get_embedder(embedder_kind))
        self.build_seconds = round(time.perf_counter() - t0, 2)

    def stats(self) -> dict:
        out = self.index.stats()
        out["build_seconds"] = self.build_seconds
        return out

    def retrieve(self, question: str, k: int = DEFAULT_TOP_K) -> List[dict]:
        hits = []
        for rank, (idx, score) in enumerate(self.index.search(question, k=k), start=1):
            c = self.chunks[idx]
            hits.append(
                {
                    "rank": rank,
                    "score": round(score, 4),
                    "chunk_id": c.chunk_id,
                    "doc_id": c.doc_id,
                    "title": c.title,
                    "heading": c.heading,
                    "department": c.department,
                    "sensitivity": c.sensitivity,
                    "source_name": c.source_name,
                    "raw_text": c.raw_text,
                }
            )
        return hits

    def answer(
        self,
        question: str,
        role: str = "employee",
        k: int = DEFAULT_TOP_K,
        provider: str = "auto",
        model: str = "",
        abstain_threshold: float = DEFAULT_ABSTAIN_THRESHOLD,
    ) -> dict:
        t_start = time.perf_counter()

        t0 = time.perf_counter()
        hits = self.retrieve(question, k=k)
        retrieve_ms = (time.perf_counter() - t0) * 1000

        top_score = hits[0]["score"] if hits else 0.0

        # Crude single global cutoff. It is the only abstention mechanism the
        # baseline has, and it is one of the things the eval is meant to expose.
        if top_score < abstain_threshold:
            answer_text = ABSTAIN_TOKEN
            provider_used = "threshold"
            cited: List[int] = []
            generate_ms = 0.0
            error = None
        else:
            t0 = time.perf_counter()
            result = generate(question, hits, provider=provider, model=model)
            generate_ms = (time.perf_counter() - t0) * 1000
            answer_text = str(result["text"])
            provider_used = str(result["provider"])
            cited = list(result["cited_indices"])  # type: ignore[arg-type]
            error = result["error"]

        abstained = answer_text.strip().upper().startswith(ABSTAIN_TOKEN)
        citations = [] if abstained else [
            {key: hits[i][key] for key in
             ("chunk_id", "doc_id", "title", "heading", "department",
              "sensitivity", "score")}
            for i in cited
        ]

        allowed = ROLE_CLEARANCE.get(role, {"public", "internal"})
        over_clearance = sorted({h["doc_id"] for h in hits
                                 if h["sensitivity"] not in allowed})

        return {
            "question": question,
            "role": role,
            "answer": ("I could not find this in the knowledge base."
                       if abstained else answer_text),
            "abstained": abstained,
            "provider": provider_used,
            "top_k": k,
            "top_score": top_score,
            "citations": citations,
            "retrieved": [
                {key: h[key] for key in
                 ("rank", "score", "chunk_id", "doc_id", "title", "heading",
                  "department", "sensitivity")}
                for h in hits
            ],
            # Diagnostic only. The baseline retrieved these anyway.
            "retrieved_above_clearance": over_clearance,
            "latency_ms": round((time.perf_counter() - t_start) * 1000, 1),
            "timings_ms": {
                "retrieve": round(retrieve_ms, 1),
                "generate": round(generate_ms, 1),
            },
            "error": error,
        }

Writing src/pipeline.py


### 3.7 The two command-line entry points

In [10]:
%%writefile run_baseline.py
#!/usr/bin/env python3
"""Run one question through the baseline RAG pipeline.

Examples
--------
    python run_baseline.py --input examples/test1.txt
    python run_baseline.py --question "How many PTO days do I get?" --role employee
    python run_baseline.py --input examples/test3.txt --role contractor --show-context
"""

from __future__ import annotations

import argparse
import json
import pathlib
import sys

from src.pipeline import DEFAULT_ABSTAIN_THRESHOLD, DEFAULT_TOP_K, KnowledgeBase

ROOT = pathlib.Path(__file__).resolve().parent
BAR = "=" * 78


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Baseline RAG over the Northwind Robotics knowledge base.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    src = p.add_mutually_exclusive_group(required=True)
    src.add_argument("--input", help="path to a text file containing the question")
    src.add_argument("--question", help="the question, given inline")

    p.add_argument("--role", default="employee",
                   choices=["employee", "contractor", "manager", "hr", "executive"],
                   help="who is asking (recorded, but NOT enforced by the baseline)")
    p.add_argument("--corpus", default=str(ROOT / "data" / "corpus"))
    p.add_argument("--top-k", type=int, default=DEFAULT_TOP_K)
    p.add_argument("--embedder", default="tfidf", choices=["tfidf", "minilm", "auto"],
                   help="tfidf is offline and deterministic; minilm needs sentence-transformers")
    p.add_argument("--provider", default="auto",
                   choices=["auto", "openai", "anthropic", "gemini", "extractive"])
    p.add_argument("--model", default="", help="override the provider's default model")
    p.add_argument("--abstain-threshold", type=float,
                   default=DEFAULT_ABSTAIN_THRESHOLD)
    p.add_argument("--show-context", action="store_true",
                   help="print the retrieved passages in full")
    p.add_argument("--out", default=str(ROOT / "outputs" / "last_run.json"))
    return p.parse_args()


def main() -> int:
    args = parse_args()

    if args.input:
        path = pathlib.Path(args.input)
        if not path.is_file():
            print(f"error: input file not found: {path}", file=sys.stderr)
            return 2
        question = path.read_text(encoding="utf-8").strip()
    else:
        question = args.question.strip()

    if not question:
        print("error: empty question", file=sys.stderr)
        return 2

    print(BAR)
    print("ENTERPRISE KNOWLEDGE BASE - BASELINE RAG")
    print(BAR)

    kb = KnowledgeBase(args.corpus, embedder_kind=args.embedder)
    st = kb.stats()
    print(f"corpus     : {st['documents']} documents -> {st['chunks']} chunks "
          f"(mean {st['mean_chunk_chars']} chars)")
    print(f"embedder   : {st['embedder_detail']}  [{st['dims']} dims]")
    print(f"index build: {st['build_seconds']}s\n")

    print(f"ROLE       : {args.role}")
    print(f"QUESTION   : {question}\n")

    res = kb.answer(
        question,
        role=args.role,
        k=args.top_k,
        provider=args.provider,
        model=args.model,
        abstain_threshold=args.abstain_threshold,
    )

    print(f"RETRIEVED TOP-{res['top_k']}")
    print("-" * 78)
    for h in res["retrieved"]:
        flag = "  <-- ABOVE ASKER'S CLEARANCE" if h["doc_id"] in res[
            "retrieved_above_clearance"] else ""
        print(f"  [C{h['rank']}] {h['score']:+.3f}  {h['chunk_id']:<12} "
              f"{h['department']:<12} {h['sensitivity']:<12} "
              f"{h['title'][:34]}{flag}")

    if args.show_context:
        print("\nCONTEXT PASSAGES")
        print("-" * 78)
        for h in kb.retrieve(question, k=args.top_k):
            print(f"\n[C{h['rank']}] {h['title']} > {h['heading']}")
            print(h["raw_text"])

    print(f"\nANSWER  (provider={res['provider']}, {res['latency_ms']} ms)")
    print("-" * 78)
    print(res["answer"])

    if res["citations"]:
        print("\nCITATIONS")
        print("-" * 78)
        for c in res["citations"]:
            print(f"  {c['chunk_id']:<12} {c['title']} > {c['heading']}  "
                  f"({c['sensitivity']})")
    elif res["abstained"]:
        print("\n[abstained - top score "
              f"{res['top_score']:.3f} < threshold {args.abstain_threshold}]")

    if res["retrieved_above_clearance"]:
        print("\n!! ACCESS-CONTROL LEAK: retrieved "
              f"{', '.join(res['retrieved_above_clearance'])} which a "
              f"'{args.role}' is not cleared to read.")
        print("   The baseline has no ACL filter. This is a known gap, not a surprise.")

    if res["error"]:
        print(f"\n[note] {res['error']}")

    out_path = pathlib.Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(res, indent=2), encoding="utf-8")
    try:
        shown = out_path.resolve().relative_to(ROOT)
    except ValueError:
        shown = out_path
    print(f"\nfull JSON written to {shown}")
    print(BAR)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing run_baseline.py


In [11]:
%%writefile run_eval.py
#!/usr/bin/env python3
"""Score the baseline on the gold question set.

    python run_eval.py
    python run_eval.py --provider extractive --limit 10

Metrics
-------
retrieval_hit@k     answerable questions where at least one gold document was retrieved
retrieval_recall    mean fraction of a question's gold documents that were retrieved
key_fact_coverage   mean fraction of the required facts that appear in the answer
answer_correct      answerable questions where every required fact appears
abstain_correct     unanswerable/restricted questions where the system abstained
false_abstain       answerable questions the system refused anyway
acl_leaks           questions where a document above the asker's clearance was retrieved

Fact checking is exact substring matching against a short list of required
strings per question. It is crude, but it is deterministic and free, which is
what a baseline needs. An LLM-as-judge scorer is Phase 2 work.
"""

from __future__ import annotations

import argparse
import json
import pathlib
import re
import statistics
import time

from src.pipeline import DEFAULT_ABSTAIN_THRESHOLD, DEFAULT_TOP_K, KnowledgeBase

ROOT = pathlib.Path(__file__).resolve().parent


def normalise(text: str) -> str:
    """Lowercase and squash whitespace so '16  weeks' matches '16 weeks'."""
    return re.sub(r"\s+", " ", text.lower())


def fact_present(fact: str, answer: str) -> bool:
    return normalise(fact) in normalise(answer)


def load_gold(path: pathlib.Path, limit: int = 0) -> list:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()]
    return rows[:limit] if limit else rows


def main() -> int:
    p = argparse.ArgumentParser(description="Evaluate the baseline RAG pipeline.")
    p.add_argument("--corpus", default=str(ROOT / "data" / "corpus"))
    p.add_argument("--gold", default=str(ROOT / "data" / "gold" / "questions.jsonl"))
    p.add_argument("--top-k", type=int, default=DEFAULT_TOP_K)
    p.add_argument("--embedder", default="tfidf", choices=["tfidf", "minilm", "auto"],
                   help="tfidf is offline and deterministic; minilm needs sentence-transformers")
    p.add_argument("--provider", default="auto",
                   choices=["auto", "openai", "anthropic", "gemini", "extractive"])
    p.add_argument("--abstain-threshold", type=float,
                   default=DEFAULT_ABSTAIN_THRESHOLD)
    p.add_argument("--limit", type=int, default=0)
    p.add_argument("--out", default=str(ROOT / "outputs" / "eval_report.json"))
    args = p.parse_args()

    gold = load_gold(pathlib.Path(args.gold), args.limit)
    kb = KnowledgeBase(args.corpus, embedder_kind=args.embedder)
    st = kb.stats()

    print("=" * 92)
    print("BASELINE EVALUATION - Enterprise Knowledge Base RAG")
    print("=" * 92)
    print(f"corpus   : {st['documents']} docs / {st['chunks']} chunks    "
          f"embedder: {st['embedder']} ({st['dims']}d)    top_k: {args.top_k}")
    print(f"questions: {len(gold)}    abstain threshold: {args.abstain_threshold}")
    print("-" * 92)
    print(f"{'QID':<5}{'TYPE':<16}{'ROLE':<11}{'RETR':<7}{'FACTS':<8}"
          f"{'ABSTAIN':<9}{'ACL':<6}{'ms':>7}  VERDICT")
    print("-" * 92)

    rows, latencies = [], []
    t_wall = time.perf_counter()

    for g in gold:
        res = kb.answer(
            g["question"], role=g.get("role", "employee"), k=args.top_k,
            provider=args.provider, abstain_threshold=args.abstain_threshold,
        )
        retrieved_docs = {h["doc_id"] for h in res["retrieved"]}
        gold_docs = set(g.get("gold_doc_ids", []))
        forbidden = set(g.get("forbidden_doc_ids", []))

        recall = (len(gold_docs & retrieved_docs) / len(gold_docs)) if gold_docs else None
        hit = bool(gold_docs & retrieved_docs) if gold_docs else None

        facts = g.get("key_facts", [])
        found = [f for f in facts if fact_present(f, res["answer"])]
        coverage = (len(found) / len(facts)) if facts else None

        leaked = sorted(forbidden & retrieved_docs) or res["retrieved_above_clearance"]
        should_abstain = bool(g.get("should_abstain"))
        abstained = bool(res["abstained"])

        if should_abstain:
            ok = abstained and not leaked
            verdict = "PASS" if ok else ("FAIL leaked" if leaked else "FAIL answered anyway")
        else:
            ok = bool(facts) and len(found) == len(facts) and not abstained
            if abstained:
                verdict = "FAIL false abstain"
            elif ok:
                # Answer is right, but the retriever still pulled a document the
                # asker is not cleared to read. Flagged, not silently passed.
                verdict = "PASS (acl leak)" if leaked else "PASS"
            else:
                missing = [f for f in facts if f not in found]
                verdict = f"FAIL missing {missing}"

        latencies.append(res["latency_ms"])
        rows.append({
            "qid": g["qid"], "type": g["type"], "role": g.get("role", "employee"),
            "question": g["question"], "gold_doc_ids": sorted(gold_docs),
            "retrieved_doc_ids": sorted(retrieved_docs), "retrieval_hit": hit,
            "retrieval_recall": recall, "key_facts": facts, "facts_found": found,
            "key_fact_coverage": coverage, "should_abstain": should_abstain,
            "abstained": abstained, "acl_leak_docs": leaked, "passed": bool(ok),
            "verdict": verdict, "latency_ms": res["latency_ms"],
            "answer": res["answer"],
        })

        print(f"{g['qid']:<5}{g['type']:<16}{g.get('role','employee'):<11}"
              f"{('-' if hit is None else ('hit' if hit else 'MISS')):<7}"
              f"{('-' if coverage is None else f'{coverage:.0%}'):<8}"
              f"{('yes' if abstained else 'no'):<9}"
              f"{('LEAK' if leaked else 'ok'):<6}"
              f"{res['latency_ms']:>7.0f}  {verdict}")

    wall = time.perf_counter() - t_wall

    answerable = [r for r in rows if not r["should_abstain"]]
    refusals = [r for r in rows if r["should_abstain"]]
    hits = [r for r in answerable if r["retrieval_hit"] is not None]

    def mean(xs):
        return round(statistics.fmean(xs), 3) if xs else 0.0

    summary = {
        "questions": len(rows),
        "answerable": len(answerable),
        "should_abstain": len(refusals),
        "retrieval_hit_at_k": mean([1.0 if r["retrieval_hit"] else 0.0 for r in hits]),
        "retrieval_recall": mean([r["retrieval_recall"] for r in hits]),
        "key_fact_coverage": mean([r["key_fact_coverage"] for r in answerable
                                   if r["key_fact_coverage"] is not None]),
        "answer_correct": mean([1.0 if r["passed"] else 0.0 for r in answerable]),
        "abstain_correct": mean([1.0 if r["passed"] else 0.0 for r in refusals]),
        "false_abstain": sum(1 for r in answerable if r["abstained"]),
        "acl_leaks": sum(1 for r in rows if r["acl_leak_docs"]),
        "overall_pass_rate": mean([1.0 if r["passed"] else 0.0 for r in rows]),
        "latency_p50_ms": round(statistics.median(latencies), 1),
        "latency_p95_ms": round(sorted(latencies)[max(0, int(0.95 * len(latencies)) - 1)], 1),
        "wall_seconds": round(wall, 1),
        "config": {
            "embedder": st["embedder"], "dims": st["dims"], "top_k": args.top_k,
            "abstain_threshold": args.abstain_threshold,
            "provider": rows and "mixed" or args.provider,
            "documents": st["documents"], "chunks": st["chunks"],
        },
    }

    print("-" * 92)
    print("SUMMARY")
    print("-" * 92)
    print(f"  retrieval hit@{args.top_k}      {summary['retrieval_hit_at_k']:.1%}"
          f"   ({len(hits)} answerable questions)")
    print(f"  retrieval recall       {summary['retrieval_recall']:.1%}")
    print(f"  key-fact coverage      {summary['key_fact_coverage']:.1%}")
    print(f"  answer correct         {summary['answer_correct']:.1%}"
          f"   ({sum(1 for r in answerable if r['passed'])}/{len(answerable)})")
    print(f"  correct abstentions    {summary['abstain_correct']:.1%}"
          f"   ({sum(1 for r in refusals if r['passed'])}/{len(refusals)})")
    print(f"  false abstentions      {summary['false_abstain']}")
    print(f"  ACCESS-CONTROL LEAKS   {summary['acl_leaks']}  "
          f"<-- baseline has no ACL filter")
    print(f"  overall pass rate      {summary['overall_pass_rate']:.1%}")
    print(f"  latency p50 / p95      {summary['latency_p50_ms']:.0f} ms / "
          f"{summary['latency_p95_ms']:.0f} ms      total {summary['wall_seconds']}s")
    print("=" * 92)

    out = pathlib.Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({"summary": summary, "results": rows}, indent=2),
                   encoding="utf-8")
    try:
        shown = out.resolve().relative_to(ROOT)
    except ValueError:
        shown = out
    print(f"report written to {shown}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing run_eval.py


## 4. Test case 1 — a single-hop lookup (this one works)

**Input:** `examples/test1.txt` — *How many days of paid time off do full-time
employees get each year?*

**Expected:** the answer contains "20 days" and cites the Paid Time Off section of
the employee handbook (HR-001).

In [12]:
!python3 run_baseline.py --input examples/test1.txt

ENTERPRISE KNOWLEDGE BASE - BASELINE RAG


corpus     : 11 documents -> 56 chunks (mean 272.4 chars)


embedder   : TF-IDF 1-2 grams + TruncatedSVD -> 55-dim LSA vectors (offline, deterministic)  [55 dims]
index build: 1.64s

ROLE       : employee
QUESTION   : How many days of paid time off do full-time employees get each year?

RETRIEVED TOP-4
------------------------------------------------------------------------------
  [C1] +0.826  HR-001#0     HR           internal     Employee Handbook - Time Off and L
  [C2] +0.470  HR-001#4     HR           internal     Employee Handbook - Time Off and L
  [C3] +0.444  HR-001#1     HR           internal     Employee Handbook - Time Off and L
  [C4] +0.283  HR-001#2     HR           internal     Employee Handbook - Time Off and L

ANSWER  (provider=extractive, 2.0 ms)
------------------------------------------------------------------------------
Full-time employees accrue **20 days of PTO per calendar year**, credited monthly at 1.67 days per completed month of service. [C1] Employees receive **10 paid sick days per year**, separate from PTO. [

**What happened.** Correct fact, correct citation — and two unrelated sentences
about sick leave and jury duty appended to it, because the extractive generator
ranks sentences by similarity to the question rather than by whether they answer
it.

## 5. Test case 2 — a multi-hop question (partial failure)

**Input:** `examples/test2.txt` — *A production database is down at 2am and the
on-call DBA is not answering pages. What is the escalation path, and who has to
approve emergency production database access?*

**Expected:** escalation through the Platform Engineering Manager (IT-001) **and**
the two-approver break-glass rule — on-call Engineering Manager plus the Security
Duty Officer (SEC-001).

In [13]:
!python3 run_baseline.py --input examples/test2.txt

ENTERPRISE KNOWLEDGE BASE - BASELINE RAG


corpus     : 11 documents -> 56 chunks (mean 272.4 chars)
embedder   : TF-IDF 1-2 grams + TruncatedSVD -> 55-dim LSA vectors (offline, deterministic)  [55 dims]
index build: 1.54s

ROLE       : employee
QUESTION   : A production database is down at 2am and the on-call DBA is not answering pages. What is the escalation path, and who has to approve emergency production database access?

RETRIEVED TOP-4
------------------------------------------------------------------------------
  [C1] +0.651  IT-001#1     IT           internal     Production Incident Response Runbo
  [C2] +0.650  IT-001#2     IT           internal     Production Incident Response Runbo
  [C3] +0.626  SEC-001#5    Security     internal     Information Security Policy
  [C4] +0.545  SEC-001#6    Security     internal     Information Security Policy

ANSWER  (provider=extractive, 2.0 ms)
------------------------------------------------------------------------------
If the DBA on call is unreachable after two pages, the in

**What happened.** Watch the retrieval block: both `IT-001` and `SEC-001` are in
the top 4, so retrieval succeeded. The answer still never says the Security Duty
Officer has to co-approve. **Using what was retrieved is what failed** — which is
precisely the job of the planner and verifier agents in Phase 2.

## 6. Test case 3 — access control (the critical failure)

**Input:** `examples/test3.txt` — *What is the salary band for a Senior Software
Engineer in Phoenix?*, asked by a **contractor**.

**Expected:** a refusal. `HR-002` is classified `confidential`, and the contractor
policy (HR-003 §2) says contractors may not be granted access to confidential
material under any circumstances.

In [14]:
!python3 run_baseline.py --input examples/test3.txt --role contractor

ENTERPRISE KNOWLEDGE BASE - BASELINE RAG


corpus     : 11 documents -> 56 chunks (mean 272.4 chars)
embedder   : TF-IDF 1-2 grams + TruncatedSVD -> 55-dim LSA vectors (offline, deterministic)  [55 dims]
index build: 1.01s

ROLE       : contractor
QUESTION   : What is the salary band for a Senior Software Engineer in Phoenix?

RETRIEVED TOP-4
------------------------------------------------------------------------------
  [C1] +0.871  HR-002#1     HR           confidential Compensation Bands FY26 (CONFIDENT  <-- ABOVE ASKER'S CLEARANCE
  [C2] +0.321  HR-002#2     HR           confidential Compensation Bands FY26 (CONFIDENT  <-- ABOVE ASKER'S CLEARANCE
  [C3] +0.242  IT-001#3     IT           internal     Production Incident Response Runbo
  [C4] +0.238  IT-002#4     IT           internal     IT Onboarding, Accounts and Equipm

ANSWER  (provider=extractive, 1.7 ms)
------------------------------------------------------------------------------
| Level | Title | Band minimum | Band midpoint | Band maximum | Target bonus | | ------

**What happened.** The baseline printed the entire confidential compensation
table to a contractor, and cited it. There is no authorisation anywhere in the
retrieval path — `--role` is recorded and ignored.

This is the single most important finding in the baseline. Phase 2 applies the
access filter **before ranking**, so a chunk the asker cannot read is never
scored, never retrieved, and never reaches a context window.

## 7. Test case 4 — a question the corpus cannot answer

**Input:** `examples/test4.txt` — *Which health insurance carrier does the company
use for US employees?*

**Expected:** "I don't know." That fact is nowhere in the corpus.

In [15]:
!python3 run_baseline.py --input examples/test4.txt

ENTERPRISE KNOWLEDGE BASE - BASELINE RAG


corpus     : 11 documents -> 56 chunks (mean 272.4 chars)
embedder   : TF-IDF 1-2 grams + TruncatedSVD -> 55-dim LSA vectors (offline, deterministic)  [55 dims]
index build: 1.06s

ROLE       : employee
QUESTION   : Which health insurance carrier does the company use for US employees?

RETRIEVED TOP-4
------------------------------------------------------------------------------
  [C1] +0.687  HR-003#3     HR           internal     Contractor and Contingent Worker P
  [C2] +0.470  HR-001#1     HR           internal     Employee Handbook - Time Off and L
  [C3] +0.296  ENG-001#0    Engineering  internal     Engineering Onboarding Guide
  [C4] +0.291  IT-001#1     IT           internal     Production Incident Response Runbo

ANSWER  (provider=extractive, 3.8 ms)
------------------------------------------------------------------------------
Employees receive **10 paid sick days per year**, separate from PTO. [C2] Contractors are not eligible for PTO, paid parental leave, sick leave, or th

**What happened.** It answered anyway, with sick-leave and contractor-benefit
text, and cited it. The only guardrail the baseline has is a single global cosine
threshold (0.25), and it never fires when it should.

## 8. Full evaluation — all 32 gold questions

`run_eval.py` scores retrieval, key-fact coverage, abstention and access-control
leakage, and writes `outputs/eval_report.json` containing every retrieved document
id and every answer, so any number below traces back to a specific question.

In [16]:
!python3 run_eval.py

BASELINE EVALUATION - Enterprise Knowledge Base RAG
corpus   : 11 docs / 56 chunks    embedder: tfidf (55d)    top_k: 4
questions: 32    abstain threshold: 0.25
--------------------------------------------------------------------------------------------
QID  TYPE            ROLE       RETR   FACTS   ABSTAIN  ACL        ms  VERDICT
--------------------------------------------------------------------------------------------
Q01  single_hop      employee   hit    100%    no       ok          2  PASS
Q02  single_hop      employee   hit    50%     no       ok          1  FAIL missing ['30 days']
Q03  single_hop      employee   hit    50%     no       ok          1  FAIL missing ['90']


Q04  single_hop      employee   hit    0%      no       ok          1  FAIL missing ['5%', 'data loss']
Q05  single_hop      employee   hit    100%    no       ok          1  PASS
Q06  single_hop      employee   hit    100%    no       ok          1  PASS
Q07  single_hop      employee   hit    100%    no       ok          1  PASS
Q08  single_hop      employee   hit    100%    no       ok          1  PASS
Q09  single_hop      employee   hit    100%    no       ok          1  PASS
Q10  single_hop      employee   hit    0%      no       ok          1  FAIL missing ['December 18', 'January 2']
Q11  single_hop      employee   hit    100%    no       LEAK        1  PASS (acl leak)
Q12  single_hop      employee   hit    0%      yes      ok          1  FAIL false abstain
Q13  single_hop      employee   hit    100%    no       ok          1  PASS
Q14  single_hop      employee   hit    100%    no       LEAK        1  PASS (acl leak)
Q15  single_hop      employee   hit    50%     no       ok     

## 9. Reading the results

```
retrieval hit@4         100.0%    (24 answerable questions)
retrieval recall        100.0%
key-fact coverage        68.1%
answer correct           50.0%    (12/24)
correct abstentions       0.0%    (0/8)
false abstentions           1
ACCESS-CONTROL LEAKS        7     <-- baseline has no ACL filter
overall pass rate        37.5%
latency p50 / p95        ~1 ms / ~2 ms
```

**Retrieval is not the bottleneck yet.** With 11 documents the right one is almost
always in the top 4. Everything downstream is the problem:

* Answers are only half correct because the generator picks locally-similar
  sentences rather than the sentence that answers the question.
* The system **never once abstained** on a question it could not answer.
* It leaked the confidential compensation document on **7 of 32 questions**,
  including to a contractor who asked for it directly.

Key-fact coverage (68%) sits well above the fully-correct rate (50%), so most
wrong answers are partly right — they find one required fact and miss the other.

### Honest caveats

* Fact checking is exact substring matching. It over-credits an answer that dumps
  a table containing the right string without actually answering, so **50% correct
  is an upper bound**. An LLM-as-judge scorer is Phase 2 work.
* 11 documents and 56 chunks is small. The retrieval numbers are optimistic and I
  expect them to fall as the corpus grows to ~40 documents.
* The no-key extractive generator is not a language model. Setting
  `OPENAI_API_KEY`, `ANTHROPIC_API_KEY` or `GOOGLE_API_KEY` switches generation to
  a real model with no other change.

---

## 10. What comes next

```
question + role
      |
      v
[ Planner ]------- decomposes into sub-questions, picks which domains to search
      |
      +--> [ HR retriever ]        \
      +--> [ IT / Eng retriever ]   >  each: ACL filter -> hybrid BM25 + dense -> rerank
      +--> [ Security retriever ]  /
      |
      v
[ Verifier ] ----- checks every claim against its cited passage; unsupported
      |            claims are dropped and the sub-question is re-sent
      v
[ Composer ] ----- merges verified claims into one cited answer, or abstains
```

Built on LangGraph, and scored by **this same `run_eval.py` against this same gold
set**, so every improvement is attributable to the component that changed.

| Metric | Baseline | Phase 2 target |
|--------|----------|----------------|
| Retrieval hit@4 | 100% | ≥ 95% at ~40 docs |
| Answer correct | 50% | ≥ 80% |
| Key-fact coverage | 68% | ≥ 90% |
| Correct abstentions | 0% (0/8) | ≥ 85% |
| **Access-control leaks** | **7 / 32** | **0** |
| Latency p50 | 1 ms | < 3 s with an LLM |

If I had to pick one number, it is access-control leaks going to zero. A knowledge
base that leaks compensation data is unusable however accurate it is, and it is
the one failure a better prompt cannot fix.